In [ ]:
import torch
from torch.utils.data import DataLoader, TensorDataset, IterableDataset
import sys
print(torch.cuda.is_available())
sys.path.append("/mnt/home/lserrano/disco-ball/")

import numpy as np
import random
import matplotlib.pyplot as plt
import h5py
import os
from tqdm import tqdm
import torch
import torch.nn as nn
from torch.optim.lr_scheduler import CosineAnnealingLR # A more standard scheduler
from einops import rearrange # A more standard import
from tqdm import tqdm
import torch.nn.functional as F

In [ ]:
from models import DISCOHouse
from advection_diffusion import Fractaloid
from train import DISCOLitModule, advection_diffusion_analytical
from plot_dataset_samples import plot_prediction_vs_ground_truth

In [ ]:
class RelativeL2(nn.Module):
    def forward(self, x, y, aggregate="mean"):
        x = rearrange(x, "b ... -> b (...)")
        y = rearrange(y, "b ... -> b (...)")
        diff_norms = torch.linalg.norm(x - y, ord=2, dim=-1)
        y_norms = torch.linalg.norm(y, ord=2, dim=-1)

        if aggregate == "mean":
            return (diff_norms / y_norms).mean()
        else:
            return (diff_norms / y_norms)

In [ ]:
def autoregressive_predict(model, initial_seq, n_pred, device):
    preds = []
    current = initial_seq.clone().to(device)
    n_input = current.shape[1]
    for t in range(n_pred):
        inp = current[:, -n_input:].to(device)
        with torch.no_grad():
            state_labels = torch.tensor([0], device=inp.device)
            next_frame, metadata = model(inp, state_labels, n_future_steps=1)
            if t == 0:
                theta = metadata['theta_latent']
        current = torch.cat([current, next_frame], axis=1)
        preds.append(next_frame)
    return torch.cat(preds, axis=1), theta

In [ ]:
class TemporalBatchDatasetFly(IterableDataset):
    def __init__(self, n_batches, batch_size, sub_x, sub_t, split="train", input_frames=16, output_frames=2,
                 L=16.0, nx=256, nt=100, T=10.0,
                 v_range=(0.01, 1.0), D_range=(0.01, 1.0),
                 fractal_degree=8, fractal_power=2, seed=None):
        self.n_batches = n_batches
        self.batch_size = batch_size
        self.sub_x = sub_x
        self.sub_t = sub_t
        self.split = split
        self.input_frames = input_frames
        self.output_frames = output_frames
        self.L = L
        self.nx = nx
        self.nt = nt
        self.T = T
        self.v_range = v_range
        self.D_range = D_range
        self.fractal_degree = fractal_degree
        self.fractal_power = fractal_power
        self.seed = seed
        self.rng = np.random.default_rng(seed)

    def __iter__(self):
        for _ in range(self.n_batches):
            input_frames = self.input_frames
            batch_inputs = []
            batch_targets = []
            batch_v = []
            batch_d = []
            batch_init = []
            for _ in range(self.batch_size):
                # Sample advection speed and viscosity
                if self.split == 'train':
                    if random.random() < 0.5:
                        v = self.rng.uniform(*self.v_range) if isinstance(self.v_range, (tuple, list)) else float(self.v_range)
                        D = 0
                    else:
                        v = 0
                        D = self.rng.uniform(*self.D_range) if isinstance(self.D_range, (tuple, list)) else float(self.D_range)
                else:
                    v = self.rng.uniform(*self.v_range) if isinstance(self.v_range, (tuple, list)) else float(self.v_range)
                    D = self.rng.uniform(*self.D_range) if isinstance(self.D_range, (tuple, list)) else float(self.D_range)
                # Generate fractaloid initial condition
                fractaloid = Fractaloid(
                    degree=self.fractal_degree,
                    power=self.fractal_power,
                    size=self.nx,
                    patch_size=self.nx
                )
                u0 = fractaloid.generate(batch_size=1, seed=None).squeeze(0).numpy()
                u0 = (u0 - u0.mean()) / (u0.std() + 1e-8)
                u_xt, x, t = advection_diffusion_analytical(
                    u0, L=self.L, v=v, D=D, nt=self.nt, T=self.T
                )
                u_xt = u_xt[::self.sub_t, ::self.sub_x]
                max_start_index_input = u_xt.shape[0] - input_frames
                input = u_xt[:input_frames].copy()
                target = u_xt[input_frames: input_frames + self.output_frames].copy()
                batch_inputs.append(torch.from_numpy(input).unsqueeze(-2).float())
                batch_targets.append(torch.from_numpy(target).unsqueeze(-2).float())
                batch_v.append(v)
                batch_d.append(D)
                batch_init.append(torch.from_numpy(u0))
            batch = {
                'input': torch.stack(batch_inputs),
                'target': torch.stack(batch_targets),
                'velocities': batch_v,
                'diffusivities': batch_d,
                'initial_conditions': torch.stack(batch_init)
            }
            yield batch

In [ ]:
batch_size=128
sub_x=1
sub_t=1
n_input_frames=16
n_output_frames=100-16 #50-n_input_frames
relative_l2_error = RelativeL2()

In [ ]:
n_batches = int(10000//batch_size)  # or set as needed for your epoch size
split="test"
train_ds = TemporalBatchDatasetFly(
    n_batches=n_batches,
    batch_size=batch_size,
    sub_x=sub_x,
    sub_t=sub_t,
    split=split,
    input_frames=n_input_frames,
    output_frames=n_output_frames,
    L=16.0,
    nx=256,
    nt=100,
    T=10.0,
    fractal_power=2.0,
    fractal_degree=256, # nx
    v_range=(0.01, 1.0),
    D_range=(0.001, 1.0),
)
train_loader = DataLoader(train_ds, batch_size=None, num_workers=4, prefetch_factor=4, pin_memory=True)

In [ ]:
#theta_path = "/mnt/home/lserrano/disco-ball/results/advection_diffusion/dense"
device="cuda" if torch.cuda.is_available() else "cpu"
#ckpt_time="2025-06-24/09-30-29" # full space of advection x diffusion
ckpt_time="2025-06-26/23-38-14" #14-28-20"#13-33-51" 
ckpt_path = f"/mnt/home/lserrano/disco-ball/outputs/{ckpt_time}/model_final.ckpt"
#theta_path = "/mnt/home/lserrano/disco-ball/results/advection_diffusion/dense"
print(f"Loading model from {ckpt_path}...")
model = DISCOLitModule.load_from_checkpoint(ckpt_path, map_location=device)
model = model.model.to(device)
model.eval()

# II. manual composition: i.e. what we want to reach by optimization

In [ ]:
class TemporalDatasetFixedCI(torch.utils.data.Dataset):
    def __init__(self, n_batches, batch_size, sub_x, sub_t, split="train", input_frames=16, output_frames=2,
                 L=16.0, nx=256, nt=100, T=10.0,
                 v_range=[0.01, 0.025, 0.05, 0.1, 0.5, 1.0], D_range=[0.001, 0.005, 0.01, 0.05, 0.1, 0.5, 1.0],
                 fractal_degree=8, fractal_power=2, seed=None):
        self.n_batches = n_batches
        self.batch_size = batch_size
        self.sub_x = sub_x
        self.sub_t = sub_t
        self.split = split
        self.input_frames = input_frames
        self.output_frames = output_frames
        self.L = L
        self.nx = nx
        self.nt = nt
        self.T = T
        self.v_range = v_range
        self.D_range = D_range
        self.fractal_degree = fractal_degree
        self.fractal_power = fractal_power
        self.seed = seed
        self.rng = np.random.default_rng(seed)
        self.u0 = []
        for _ in range(self.batch_size):
            fractaloid = Fractaloid(
                degree=self.fractal_degree,
                power=self.fractal_power,
                size=self.nx,
                patch_size=self.nx
            )
            u0 = fractaloid.generate(batch_size=1, seed=None).squeeze(0).numpy()
            u0 = (u0 - u0.mean()) / (u0.std() + 1e-8)
            self.u0.append(torch.from_numpy(u0))
            
        self.u0 = torch.stack(self.u0)

    def __len__(self):
        return len(self.u0)

    def __getitem__(self, idx):
    
        input_frames = self.input_frames
        batch_inputs = []
        batch_targets = []
        batch_v = []
        batch_d = []
        batch_init = []
        for v in self.v_range:
            for d in self.D_range:
                u0 = self.u0[idx]
                u_xt, x, t = advection_diffusion_analytical(
                    u0, L=self.L, v=v, D=d, nt=self.nt, T=self.T
                )
                u_xt = u_xt[::self.sub_t, ::self.sub_x]
                input = u_xt[:input_frames].copy()
                target = u_xt[input_frames: input_frames + self.output_frames].copy()
                
                batch_inputs.append(torch.from_numpy(input).unsqueeze(-2).float())
                batch_targets.append(torch.from_numpy(target).unsqueeze(-2).float())
                batch_v.append(v)
                batch_d.append(d)
                batch_init.append(u0)
                
        batch = {
            'input': torch.stack(batch_inputs),
            'target': torch.stack(batch_targets),
            'velocities': batch_v,
            'diffusivities': batch_d,
            'initial_conditions': torch.stack(batch_init)
        }
        return batch

In [ ]:
n_batches = 1  # or set as needed for your epoch size
batch_size = 1
split="test"
n_input_frames=16
n_output_frames=34

#advection_speeds = [0.4, 0.4, 0.8]
#viscosities = [0.0, 0, 0.]
advection_speeds = [0.2, 0., 0.2]
viscosities = [0.0, 0.4, 0.4]

all_velocities = []
all_diffusivities = []
all_input = []
all_target = []
all_theta_latent = []
all_theta = []

train_ds = TemporalDatasetFixedCI(
        n_batches=n_batches,
        batch_size=batch_size,
        sub_x=sub_x,
        sub_t=sub_t,
        split=split,
        input_frames=n_input_frames,
        output_frames=n_output_frames,
        L=16.0,
        nx=256,
        nt=100,
        T=10.0,
        fractal_power=2.0,
        fractal_degree=256, # nx
        v_range=[0],#(0.01, 1.0),
        D_range=[0], #(0.001, 1.0),
    )

for advection_speed, viscosity in zip(advection_speeds, viscosities):

    train_ds.v_range=[advection_speed]
    train_ds.D_range=[viscosity]
    train_loader = DataLoader(train_ds, batch_size=batch_size, num_workers=1, prefetch_factor=1, pin_memory=True, shuffle=False)
    
    for batch in tqdm(train_loader):
        inp, target = batch["input"], batch["target"]
        inp = inp.squeeze(1)
        target = target.squeeze(1)
        all_velocities += batch["velocities"]
        all_diffusivities += batch["diffusivities"]
        
    inp = inp.to(device)
    target = target.to(device)
    state_labels = torch.tensor([0], device=inp.device)
    
    x_shape = inp.shape
    B, T, C = x_shape[:3]
    spatial = x_shape[3:]
    dim = len(spatial)
    
    n_sample = inp.shape[0]
    #pred, theta = autoregressive_predict(model, inp, n_pred=target.shape[1], device=device)

    with torch.no_grad():
        # encode into 2 dimensional
        theta_latent, metadata= model.encode_theta_latent(inp, state_labels)
        
        # decode into 100k parameters
        theta = model.decode_theta(theta_latent, dim)
        
        #predict
        pred, metadata = model.solve_ode(inp[:, -1], theta, state_labels, dim, n_future_steps=1, predict_normed=False, metadata=metadata)
    rollout_error = relative_l2_error(pred[:, -1:], target[:, :1]).item()
    
    print(f"Initial error", rollout_error)

    all_theta_latent.append(theta_latent)
    all_theta.append(theta)
    all_input.append(inp)
    all_target.append(target)


all_theta_latent = torch.stack(all_theta_latent)
all_theta = torch.stack(all_theta)
all_input = torch.stack(all_input)
all_target = torch.stack(all_target)

In [ ]:
### setup the data 
theta1 = all_theta[0]
theta2 = all_theta[1]

theta_latent1 = all_theta_latent[0]
theta_latent2 = all_theta_latent[0]

x_test = all_input[2]
y_test = all_target[2]
state_labels = torch.tensor([0], device=x_test.device)

In [ ]:
model.max_steps=1

In [ ]:
pred_test = []
x_test_ = x_test[:, -1].clone()
n_output_frames=1
with torch.no_grad():
    for _ in range(n_output_frames):
        pred, _ = model.solve_ode_with_2_operators(x_test_, theta1, theta2, state_labels, dim, n_future_steps=1, predict_normed=False, metadata=metadata # Re-initialize metadata for the test run
        )
        pred_test.append(pred[:, -1:])
        x_test_ = pred[:, -1]

pred_test = torch.cat(pred_test, 1)
# Calculate the test error.
test_error = relative_l2_error(pred_test, y_test[:, :n_output_frames], aggregate=None)

In [ ]:
test_error.mean()

In [ ]:
pred_test.shape

In [ ]:
print('test_error', test_error)
idx = 0
for t in range(pred_test.shape[1]):
    plt.plot(pred_test.detach().cpu().numpy()[idx, t].squeeze(), label=t)
plt.legend()

In [ ]:
for t in range(pred_test.shape[1]):
    plt.plot(y_test.detach().cpu().numpy()[idx, t].squeeze(), label=t)
plt.legend()

In [ ]:
# --- Hyperparameters and setup ---
# Use a more descriptive variable name for num_steps.
idx_=0
epochs = 1000
n_output_frames=34


# Training data for interpolation.
#x_train = rearrange(inp[:, :-1].clone(), "b t c h -> (b t) 1 c h").to(device)
#y_train = rearrange(inp[:, 1:].clone(), "b t c h -> (b t) 1 c h").to(device)

# Test data for extrapolation.
#x_test = inp[:, -1].to(device)
#y_test = target.to(device)

# --- Optimizer and Scheduler ---
# Use standard PyTorch classes for clarity.
#theta1 = theta1.detach().clone().requires_grad_()
#theta2 = theta2.detach().clone().requires_grad_()

theta1 = (0.1*torch.randn_like(theta1.detach())).requires_grad_()
theta2 = (0.1*torch.randn_like(theta2.detach())).requires_grad_()

optimizer = torch.optim.AdamW([theta1] + [theta2], lr=1e-3)

# Use CosineAnnealingLR for a standard cosine decay schedule.
# This is a common and effective choice.
scheduler = CosineAnnealingLR(optimizer, T_max=epochs) 

print("Starting training...")
# Wrap the range in tqdm to get a progress bar.
for epoch in tqdm(range(epochs), desc="Training"):
    # --- Interpolation Step (Training) ---
    # Set the model to training mode (if applicable).
    # Some models might have different modes for training and inference.
    model.train()
    
    # Initialize metadata for the solve_ode call.
    # It's better to initialize it here if it's used within the loop.
    metadata = {} 

    t = random.randint(0, n_input_frames-2)

    pred, _ = model.solve_ode_with_2_operators(x_test[:, t], theta1, theta2, state_labels, dim, n_future_steps=1, predict_normed=False, metadata=metadata # Re-initialize metadata for the test run
    )
    
    #x_train = inp[:, t]
    #y_train = inp[:, t+1]
    
    # Run the model to get the prediction.
    
    
    # Calculate the training loss.
    loss = relative_l2_error(pred, x_test[:, t+1])# + 0.001*torch.abs(F.cosine_similarity(theta_latent1, theta_latent2, dim=1)).mean()
    
    # --- Backpropagation ---
    # A standard training step.
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    
    # Update the learning rate.
    scheduler.step()

    # --- Evaluation Step (Extrapolation) ---
    # It's a good practice to evaluate the model without gradients.
        
    # --- Print progress ---
    # Print the losses in a clear, formatted way.
    # You can print less frequently to avoid excessive output.
    if (epoch + 1) % 10 == 0 or epoch == 0:
        with torch.no_grad():
            # Set the model to evaluation mode.
            # This is important for layers like BatchNorm or Dropout.
            model.eval()

            pred_test = []
            x_test_ = x_test[:, -1].clone()
            for _ in range(n_output_frames):
                pred, _ = model.solve_ode_with_2_operators(x_test_, theta1, theta2, state_labels, dim, n_future_steps=1, predict_normed=False, metadata=metadata # Re-initialize metadata for the test run
    )
                pred_test.append(pred)
                x_test_ = pred[:, -1]

            pred_test = torch.cat(pred_test, 1)
            # Calculate the test error.
            test_error = relative_l2_error(pred_test, y_test).item()
    
            
        print(
            f"Epoch [{epoch+1}/{epochs}] |",
            f"Interpolation next time-step (Loss): {loss.item():.6f} | "
            f"Extrapolation next time-step (Error): {test_error:.6f}"
        )

print("Training finished.")

In [ ]:
# --- Hyperparameters and setup ---
# Use a more descriptive variable name for num_steps.
idx_=0
epochs = 1000
n_output_frames=34


# Training data for interpolation.
#x_train = rearrange(inp[:, :-1].clone(), "b t c h -> (b t) 1 c h").to(device)
#y_train = rearrange(inp[:, 1:].clone(), "b t c h -> (b t) 1 c h").to(device)

# Test data for extrapolation.
#x_test = inp[:, -1].to(device)
#y_test = target.to(device)

# --- Optimizer and Scheduler ---
# Use standard PyTorch classes for clarity.
#theta1 = theta1.detach().clone().requires_grad_()
#theta2 = theta2.detach().clone().requires_grad_()

theta1 = (0.1*torch.randn_like(theta1.detach())).requires_grad_()
theta2 = (0.1*torch.randn_like(theta2.detach())).requires_grad_()

optimizer = torch.optim.AdamW([theta1], lr=1e-3)

# Use CosineAnnealingLR for a standard cosine decay schedule.
# This is a common and effective choice.
scheduler = CosineAnnealingLR(optimizer, T_max=epochs) 

print("Starting training...")
# Wrap the range in tqdm to get a progress bar.
for epoch in tqdm(range(epochs), desc="Training"):
    # --- Interpolation Step (Training) ---
    # Set the model to training mode (if applicable).
    # Some models might have different modes for training and inference.
    model.train()
    
    # Initialize metadata for the solve_ode call.
    # It's better to initialize it here if it's used within the loop.
    metadata = {} 

    t = random.randint(0, n_input_frames-2)

    pred, _ = model.solve_ode(x_test[:, t], theta1, state_labels, dim, n_future_steps=1, predict_normed=False, metadata=metadata # Re-initialize metadata for the test run
    )
    
    #x_train = inp[:, t]
    #y_train = inp[:, t+1]
    
    # Run the model to get the prediction.
    
    
    # Calculate the training loss.
    loss = relative_l2_error(pred, x_test[:, t+1])# + 0.001*torch.abs(F.cosine_similarity(theta_latent1, theta_latent2, dim=1)).mean()
    
    # --- Backpropagation ---
    # A standard training step.
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    
    # Update the learning rate.
    scheduler.step()

    # --- Evaluation Step (Extrapolation) ---
    # It's a good practice to evaluate the model without gradients.
        
    # --- Print progress ---
    # Print the losses in a clear, formatted way.
    # You can print less frequently to avoid excessive output.
    if (epoch + 1) % 10 == 0 or epoch == 0:
        with torch.no_grad():
            # Set the model to evaluation mode.
            # This is important for layers like BatchNorm or Dropout.
            model.eval()

            pred_test = []
            x_test_ = x_test[:, -1].clone()
            for _ in range(n_output_frames):
                pred, _ = model.solve_ode(x_test_, theta1, state_labels, dim, n_future_steps=1, predict_normed=False, metadata=metadata # Re-initialize metadata for the test run
    )
                pred_test.append(pred)
                x_test_ = pred[:, -1]

            pred_test = torch.cat(pred_test, 1)
            # Calculate the test error.
            test_error = relative_l2_error(pred_test, y_test).item()
    
            
        print(
            f"Epoch [{epoch+1}/{epochs}] |",
            f"Interpolation next time-step (Loss): {loss.item():.6f} | "
            f"Extrapolation next time-step (Error): {test_error:.6f}"
        )

print("Training finished.")

In [ ]:
# --- Hyperparameters and setup ---
# Use a more descriptive variable name for num_steps.
idx_=0
epochs = 500
n_output_frames=34


# Training data for interpolation.
#x_train = rearrange(inp[:, :-1].clone(), "b t c h -> (b t) 1 c h").to(device)
#y_train = rearrange(inp[:, 1:].clone(), "b t c h -> (b t) 1 c h").to(device)

# Test data for extrapolation.
#x_test = inp[:, -1].to(device)
#y_test = target.to(device)

# --- Optimizer and Scheduler ---
# Use standard PyTorch classes for clarity.
#theta1 = theta1.detach().clone().requires_grad_()
#theta2 = theta2.detach().clone().requires_grad_()

#theta_latent1 = (0.01*torch.randn_like(theta_latent.detach())).requires_grad_()
#theta_latent2 = (0.01*torch.randn_like(theta_latent.detach())).requires_grad_()

theta_latent1 = (torch.randn_like(all_theta_latent[0].detach())).requires_grad_()
theta_latent2 = (torch.randn_like(all_theta_latent[1].detach())).requires_grad_()

theta_latent_1_time = []
theta_latent_2_time = []

optimizer = torch.optim.AdamW([theta_latent1] + [theta_latent2], lr=5e-1)

# Use CosineAnnealingLR for a standard cosine decay schedule.
# This is a common and effective choice.
scheduler = CosineAnnealingLR(optimizer, T_max=epochs) 

print("Starting training...")
# Wrap the range in tqdm to get a progress bar.
for epoch in tqdm(range(epochs), desc="Training"):
    # --- Interpolation Step (Training) ---
    # Set the model to training mode (if applicable).
    # Some models might have different modes for training and inference.
    model.train()
    
    # Initialize metadata for the solve_ode call.
    # It's better to initialize it here if it's used within the loop.
    metadata = {} 

    t = random.randint(0, n_input_frames-2)

    theta1 = model.decode_theta(theta_latent1, dim)
    theta2 = model.decode_theta(theta_latent2, dim)

    pred, _ = model.solve_ode_with_2_operators(x_test[:, t], theta1, theta2, state_labels, dim, n_future_steps=1, predict_normed=False, metadata=metadata # Re-initialize metadata for the test run
    )

    theta_latent_1_time.append(theta_latent1.cpu().detach())
    theta_latent_2_time.append(theta_latent2.cpu().detach())
    
    #x_train = inp[:, t]
    #y_train = inp[:, t+1]
    
    # Run the model to get the prediction.
    
    
    # Calculate the training loss.
    loss = relative_l2_error(pred, x_test[:, t+1])# + 0.001*torch.abs(F.cosine_similarity(theta_latent1, theta_latent2, dim=1)).mean()
    
    # --- Backpropagation ---
    # A standard training step.
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    
    # Update the learning rate.
    scheduler.step()

    # --- Evaluation Step (Extrapolation) ---
    # It's a good practice to evaluate the model without gradients.
        
    # --- Print progress ---
    # Print the losses in a clear, formatted way.
    # You can print less frequently to avoid excessive output.
    if (epoch + 1) % 10 == 0 or epoch == 0:
        with torch.no_grad():
            # Set the model to evaluation mode.
            # This is important for layers like BatchNorm or Dropout.
            model.eval()
            theta1 = model.decode_theta(theta_latent1, dim)
            theta2 = model.decode_theta(theta_latent2, dim)

            pred_test = []
            x_test_ = x_test[:, -1].clone()
            for _ in range(n_output_frames):
                pred, _ = model.solve_ode_with_2_operators(x_test_, theta1, theta2, state_labels, dim, n_future_steps=1, predict_normed=False, metadata=metadata # Re-initialize metadata for the test run
    )
                pred_test.append(pred)
                x_test_ = pred[:, -1]

            pred_test = torch.cat(pred_test, 1)
            # Calculate the test error.
            test_error = relative_l2_error(pred_test, y_test).item()
    
            
        print(
            f"Epoch [{epoch+1}/{epochs}] |",
            f"Interpolation next time-step (Loss): {loss.item():.6f} | "
            f"Extrapolation next time-step (Error): {test_error:.6f}"
        )

print("Training finished.")
theta_latent_1_time = torch.stack(theta_latent_1_time)
theta_latent_2_time = torch.stack(theta_latent_2_time)

In [ ]:
#0.043855

In [ ]:
# Unpack the 2D coordinates over time
x = theta_latent_1_time[:, 0, 0]
y = theta_latent_1_time[:, 0, 1]
t = np.arange(theta_latent_2_time.shape[0])  # time indices

plt.figure(figsize=(8, 6))
sc = plt.scatter(x, y, c=t, cmap="viridis", s=40, alpha=0.8)
plt.plot(x, y, color='lightgray', alpha=0.5, linewidth=1)  # optional: connect points to show trajectory

# Highlight specific point in red
highlight_x = all_theta.detach().cpu()[1, 0, 0]
highlight_y = all_theta.detach().cpu()[1, 0, 1]
#plt.scatter(highlight_x, highlight_y, c="red", s=100, edgecolor='black', label="t=1")

plt.colorbar(sc, label="Time Step")
plt.title("Latent Trajectory Over Time")
plt.xlabel("Theta[0]")
plt.ylabel("Theta[1]")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()


In [ ]:
# Unpack the 2D coordinates over time
x = theta_latent_2_time[:, 0, 0]
y = theta_latent_2_time[:, 0, 1]
t = np.arange(theta_latent_2_time.shape[0])  # time indices

plt.figure(figsize=(8, 6))
sc = plt.scatter(x, y, c=t, cmap="viridis", s=40, alpha=0.8)
plt.plot(x, y, color='lightgray', alpha=0.5, linewidth=1)  # optional: connect points to show trajectory

# Highlight specific point in red
highlight_x = all_theta.detach().cpu()[1, 0, 0]
highlight_y = all_theta.detach().cpu()[1, 0, 1]
#plt.scatter(highlight_x, highlight_y, c="red", s=100, edgecolor='black', label="t=1")

plt.colorbar(sc, label="Time Step")
plt.title("Latent Trajectory Over Time")
plt.xlabel("Theta[0]")
plt.ylabel("Theta[1]")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()


In [ ]:
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
from matplotlib import cm
import random

# --- Mock Dependencies (Updated for Clarity) ---


# --- The User's Loss Function, MODIFIED for Separate Inputs ---
def loss_fn_for_plotting(theta1_latent, theta2_latent, model, x_test, t_fixed, dim, state_labels, n_future_steps, predict_normed, metadata):
    """
    User's loss function, now taking theta1_latent and theta2_latent as separate inputs.
    """
    # No slicing needed here, as the inputs are already distinct
    theta1 = model.decode_theta(theta1_latent, dim)
    theta2 = model.decode_theta(theta2_latent, dim)

    pred, _ = model.solve_ode_with_2_operators(
        x_test[:, t_fixed], theta1, theta2, state_labels, dim, 
        n_future_steps=n_future_steps, predict_normed=predict_normed, metadata=metadata
    )
    
    loss = relative_l2_error(pred, x_test[:, t_fixed + 1], aggregate="None")

    return loss

### 2. The Unified Plotting Logic (Updated)

def plot_loss_landscape_slice(
    loss_fn, model, x_test, dim, state_labels, metadata, n_input_frames,
    varied_part: str, fixed_value: tuple = (0.0, 0.0),
    n_future_steps=1, predict_normed=False, batch_index=0):
    """
    Plots the 2D loss landscape by varying one latent vector while fixing the other.

    Args:
        varied_part (str): 'theta1' to vary theta1_latent or 'theta2' to vary theta2_latent.
        fixed_value (tuple): A 2D tuple (x, y) at which to fix the other latent vector.
    """
    if varied_part == 'theta1':
        title_part = f'Varying theta1_latent | Fixed theta2_latent at {fixed_value}'
        xlabel = 'Latent Dim 1 for Theta1'
        ylabel = 'Latent Dim 2 for Theta1'
    elif varied_part == 'theta2':
        title_part = f'Varying theta2_latent | Fixed theta1_latent at {fixed_value}'
        xlabel = 'Latent Dim 1 for Theta2'
        ylabel = 'Latent Dim 2 for Theta2'
    else:
        raise ValueError("varied_part must be 'theta1' or 'theta2'.")

    print(f"Preparing to plot landscape: {title_part}")
    
    # --- Step 1: Define the 2D Grid ---
    x_range = np.linspace(-1, 1, 50)
    y_range = np.linspace(-1, 1, 50)
    X, Y = np.meshgrid(x_range, y_range)
    theta_grid = np.stack([X, Y], axis=-1)
    theta_grid = torch.from_numpy(theta_grid).cuda().float()

    # --- Step 2: Fix the Time Step and Base Latent Vectors ---
    t_fixed = random.randint(0, n_input_frames - 2)
    print(f"Using a fixed time step t = {t_fixed}")

    if varied_part == "theta1":
        current_theta1_latent = rearrange(theta_grid, 'n1 n2 c -> (n1 n2) c')
        current_theta2_latent = fixed_value[None, ...].repeat(50*50, 1)
        
    elif varied_part == "theta2":
        current_theta2_latent = rearrange(theta_grid, 'n1 n2 c -> (n1 n2) c')
        current_theta1_latent = fixed_value[None, ...].repeat(50*50, 1)

    
    x_test = x_test.clone().repeat(50*50,1,1,1)
        
    # --- Step 3: Iterate and compute loss ---
    print("Computing loss values over the grid...")

    with torch.no_grad():
        Z = loss_fn(
            current_theta1_latent, current_theta2_latent, model, x_test, t_fixed, dim,
            state_labels, n_future_steps, predict_normed, metadata
        )
    Z = rearrange(Z, '(n1 n2)-> n1 n2', n1=50).detach().cpu()
    min_loss_for_log = Z.min() # Find the minimum non-zero loss value
    norm = LogNorm(vmin=min_loss_for_log, vmax=Z.max())

    print("Computation complete. Now plotting the surface...")

    # --- Step 4: Plot the 3D Surface ---
    fig = plt.figure(figsize=(15, 12))
    ax = fig.add_subplot(111, projection='3d')
    surf = ax.plot_surface(X, Y, Z, cmap=cm.plasma, norm=norm, linewidth=0, antialiased=False)
    
    # Set labels and title
    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel)
    ax.set_zlabel('Loss')
    ax.set_title(f'Loss Landscape for Latent Space\n{title_part}')
    fig.colorbar(surf, shrink=0.5, aspect=5, label='Loss Value')
    ax.view_init(elev=25, azim=-55)
    plt.show()
    
    return Z

In [ ]:
from matplotlib.colors import LogNorm # <-- Import LogNorm

In [ ]:
    # --- Plot 1: Vary theta2_latent, fix theta1_latent ---
print("--- Plotting Landscape for theta2 (Fixing theta1) ---")
Z = plot_loss_landscape_slice(
    loss_fn=loss_fn_for_plotting,
    model=model,
    x_test=x_test,
    dim=dim,
    state_labels=state_labels,
    metadata=metadata,
    n_input_frames=n_input_frames,
    varied_part='theta2',
    fixed_value=all_theta_latent[0][0].cuda() # We fix theta1_latent at (0,0) for the whole batch
)

In [ ]:
Z.min()

In [ ]:
    # --- Plot 2: Vary theta1_latent, fix theta2_latent ---
print("\n--- Plotting Landscape for theta1 (Fixing theta2) ---")
Z = plot_loss_landscape_slice(
    loss_fn=loss_fn_for_plotting,
    model=model,
    x_test=x_test,
    dim=dim,
    state_labels=state_labels,
    metadata=metadata,
    n_input_frames=n_input_frames,
    varied_part='theta1',
    fixed_value=all_theta_latent[1][0] # We fix theta2_latent at (0,0)
)

print("\nPlotting complete. Two 3D plot windows should have appeared sequentially.")

In [ ]:
    # --- Plot 1: Vary theta2_latent, fix theta1_latent ---
print("--- Plotting Landscape for theta2 (Fixing theta1) ---")
Z = plot_loss_landscape_slice(
    loss_fn=loss_fn_for_plotting,
    model=model,
    x_test=x_test,
    dim=dim,
    state_labels=state_labels,
    metadata=metadata,
    n_input_frames=n_input_frames,
    varied_part='theta2',
    fixed_value=torch.randn_like(all_theta_latent[0][0]).cuda() # what if it is a random theta
)

In [ ]:
    # --- Plot 2: Vary theta1_latent, fix theta2_latent ---
print("\n--- Plotting Landscape for theta1 (Fixing theta2) ---")
Z = plot_loss_landscape_slice(
    loss_fn=loss_fn_for_plotting,
    model=model,
    x_test=x_test,
    dim=dim,
    state_labels=state_labels,
    metadata=metadata,
    n_input_frames=n_input_frames,
    varied_part='theta1',
    fixed_value=torch.randn_like(all_theta_latent[1][0]).cuda() # We fix theta2_latent at (0,0)
)

print("\nPlotting complete. Two 3D plot windows should have appeared sequentially.")

In [ ]:
Z.min()

In [ ]:
x_test.shape

# II. Test 

# III. Further analysis

## III.b Fixed initial conditions, varying parameters

In [ ]:
def loss_fn(theta):
    t = random.randint(0, n_input_frames-2)

    theta_latent1 = theta_latent[:, :2]
    theta_latent2 = theta_latent[:, 2:]

    theta1 = model.decode_theta(theta_latent1, dim)
    theta2 = model.decode_theta(theta_latent2, dim)

    pred, _ = model.solve_ode_with_2_operators(x_test[:, t], theta1, theta2, state_labels, dim, n_future_steps=1, predict_normed=False, metadata=metadata # Re-initialize metadata for the test run
    )    
    # Calculate the training loss.
    loss = relative_l2_error(pred, x_test[:, t+1])

    return loss


def second_order_update(param_tensor, loss_fn, lr=1.0, damping=1e-6, second_order=False):
    """
    Performs a single second-order optimization step on a batched parameter tensor.

    Args:
        param_tensor (torch.Tensor): The parameter tensor to optimize, with shape (B, D).
        loss_fn (callable): A function that takes the parameter tensor and returns a scalar loss.
        lr (float): Learning rate (step size).
        damping (float): Damping term for numerical stability when inverting the Hessian.
    """
    # Ensure the parameter requires a gradient
    if not param_tensor.requires_grad:
        raise ValueError("The parameter tensor must have requires_grad=True.")

    # --- Step 1: Compute the first-order gradient (dLoss / dparam) ---
    # We need to create the graph for second derivatives, so create_graph=True
    loss = loss_fn(param_tensor)
    
    first_derivative_grads = torch.autograd.grad(
        outputs=loss,
        inputs=param_tensor,
        create_graph=True,  # Crucial for computing the Hessian
        retain_graph=True
    )
    grad_L_wrt_param = first_derivative_grads[0]
    
    # --- Step 2: Compute the Hessian (d^2Loss / dparam^2) ---
    # The Hessian will be of shape (B, D, D) where D is the dimension of the parameter (2 in your case).
    # We can do this efficiently using a loop over the parameter's dimensions.


    if second_order:
        param_dim = param_tensor.shape[1]  # D = 2
        hessian_rows = []
        
        for i in range(param_dim):
            # We need to compute the gradient of the i-th component of the first derivative
            # w.r.t. the entire parameter tensor.
            # We sum over the batch dimension to make the output a scalar for autograd.
            hessian_row_i, = torch.autograd.grad(
                outputs=grad_L_wrt_param[:, i].sum(),
                inputs=param_tensor,
                retain_graph=True  # Retain graph for the next iteration of the loop
            )
            hessian_rows.append(hessian_row_i)
    
        # Stack the rows and permute dimensions to get shape (B, D, D)
        hessian_tensor = torch.stack(hessian_rows).permute(1, 0, 2)
        
        # --- Step 3: Compute the inverse of the Hessian ---
        
        # Add a small damping term to the diagonal of the Hessian for stability
        identity_matrix = torch.eye(param_dim, device=param_tensor.device, dtype=param_tensor.dtype)
        damped_hessian = hessian_tensor + damping * identity_matrix.unsqueeze(0)
        
        # Compute the inverse for each matrix in the batch
        try:
            hessian_inverse = torch.linalg.inv(damped_hessian)
        except torch.linalg.LinAlgError:
            print("Hessian is singular, skipping update.")
            return
    
        # --- Step 4: Compute the update step and update the parameter ---
        
        # Reshape gradient for batch matrix multiplication: (B, D) -> (B, D, 1)
        grad_L_wrt_param_reshaped = grad_L_wrt_param.unsqueeze(-1)
        
        # Compute the Newton step: -H_inv * grad_L
        update_step = -torch.matmul(hessian_inverse, grad_L_wrt_param_reshaped).squeeze(-1)

    else:
        update_step = -grad_L_wrt_param
        
    
    # Manually update the parameter tensor using a learning rate
    with torch.no_grad():
        param_tensor.add_(lr * update_step)

    # Clear gradients for the next iteration
    # This is important if you use this function inside a training loop with an optimizer.
    if param_tensor.grad is not None:
        param_tensor.grad.zero_()
        
    return loss.item()